# Bab 5: Klasifikasi

Klasifikasi adalah proses memprediksi label atau kategori diskrit untuk data yang diberikan.
Berbeda dengan regresi yang memprediksi nilai kontinu, klasifikasi menjawab pertanyaan "Ya/Tidak" atau "A/B/C".

Dalam bab ini, kita akan mempelajari:
1. Naive Bayes.
2. Discriminant Analysis (LDA).
3. Regresi Logistik.
4. Evaluasi Model Klasifikasi.
5. Strategi untuk Imbalanced Data.

## 1. Naive Bayes

Algoritma ini didasarkan pada Teorema Bayes dengan asumsi "naif" bahwa semua fitur bersifat independen satu sama lain.

### Teorema Bayes:
$$P(C|X) = \frac{P(X|C)P(C)}{P(X)}$$

### Kelebihan:
- Sangat cepat dan efisien.
- Bekerja dengan baik untuk data teks (seperti filter spam).
- Membutuhkan sedikit data pelatihan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Load iris data
iris = sns.load_dataset('iris')
X = iris.drop('species', axis=1)
y = iris['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)

print("Confusion Matrix Naive Bayes:")
print(confusion_matrix(y_test, y_pred))

## 2. Discriminant Analysis (LDA)

Linear Discriminant Analysis (LDA) mencoba mencari kombinasi fitur yang paling baik dalam memisahkan dua atau lebih kelas.

### Perbedaan dengan PCA:
- **PCA**: Mencari arah varians maksimum (unsupervised).
- **LDA**: Mencari arah yang memaksimalkan pemisahan kelas (supervised).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
print(f"Akurasi LDA: {lda.score(X_test, y_test):.2f}")

## 3. Regresi Logistik (Logistic Regression)

Meskipun namanya regresi, ini adalah model klasifikasi linear yang sangat kuat.

### Fungsi Sigmoid (Logit):
Ia memetakan nilai input ke rentang antara 0 dan 1, yang diinterpretasikan sebagai probabilitas.
$$f(z) = \frac{1}{1 + e^{-z}}$$

### Odds Ratio:
Rasio probabilitas sukses terhadap probabilitas gagal.
$$Odds = \frac{p}{1-p}$$

In [ ]:
from sklearn.linear_model import LogisticRegression

# Gunakan dataset biner (hanya 2 spesies)
df_binary = iris[iris['species'] != 'setosa']
X_bin = df_binary.drop('species', axis=1)
y_bin = df_binary['species']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

log_reg = LogisticRegression()
log_reg.fit(X_train_b, y_train_b)

probs = log_reg.predict_proba(X_test_b)[:, 1]
plt.figure(figsize=(10, 6))
sns.histplot(probs, bins=20, kde=True)
plt.title("Distribusi Probabilitas Prediksi Regresi Logistik")
plt.show()

## 4. Evaluasi Model Klasifikasi

Akurasi saja seringkali menyesatkan, terutama pada data yang tidak seimbang.

### A. Confusion Matrix
- **True Positive (TP)**: Benar diprediksi positif.
- **False Positive (FP)**: Salah diprediksi positif (Type I Error).
- **False Negative (FN)**: Salah diprediksi negatif (Type II Error).

### B. Metrik Kunci:
- **Precision**: TP / (TP + FP) -> Seberapa akurat saat memprediksi positif.
- **Recall (Sensitivity)**: TP / (TP + FN) -> Seberapa banyak kasus positif yang tertangkap.
- **F1-Score**: Rata-rata harmonik antara Precision dan Recall.

### C. Kurva ROC dan AUC
- **ROC Curve**: Plot True Positive Rate vs False Positive Rate pada berbagai ambang batas.
- **AUC (Area Under Curve)**: Ringkasan performa model (1.0 sempurna, 0.5 acak).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Encode label ke biner
y_test_binary = (y_test_b == 'virginica').astype(int)

fpr, tpr, thresholds = roc_curve(y_test_binary, probs)
auc = roc_auc_score(y_test_binary, probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend()
plt.show()

## 5. Strategi untuk Imbalanced Data

Terjadi ketika satu kelas jauh lebih dominan daripada yang lain (misal: deteksi penipuan bank).

### Teknik Penanganan:
- **Undersampling**: Mengurangi data kelas mayoritas.
- **Oversampling**: Menduplikasi data kelas minoritas.
- **SMOTE (Synthetic Minority Over-sampling Technique)**: Menciptakan data buatan untuk kelas minoritas.
- **Penyesuaian Ambang Batas (Thresholding)**: Mengubah ambang batas probabilitas dari 0.5 menjadi nilai lain.

In [ ]:
from imblearn.over_sampling import SMOTE

# Simulasi data imbalanced
from sklearn.datasets import make_classification
X_imb, y_imb = make_classification(n_samples=1000, n_classes=2, weights=[0.9, 0.1], random_state=42)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_imb, y_imb)

print(f"Distribusi asli: {np.bincount(y_imb)}")
print(f"Distribusi setelah SMOTE: {np.bincount(y_res)}")

## 6. Kesimpulan Bab 5

Klasifikasi adalah alat kunci untuk pengambilan keputusan otomatis.
- Gunakan Naive Bayes untuk baseline yang cepat.
- Regresi Logistik sangat bagus jika Anda butuh probabilitas dan interpretabilitas.
- Jangan pernah mengandalkan akurasi saja pada data imbalanced; gunakan Precision, Recall, dan AUC.

### Konten Tambahan Detail: Naive Bayes Lanjutan

Terdapat beberapa varian Naive Bayes:
1. **Gaussian NB**: Untuk data kontinu yang diasumsikan berdistribusi normal.
2. **Multinomial NB**: Untuk data diskrit seperti jumlah kata dalam teks.
3. **Bernoulli NB**: Untuk data biner (ada/tidak ada fitur).

#### Penjelasan Mendalam tentang Log-Odds

Dalam regresi logistik, kita sebenarnya melakukan regresi linear pada logaritma dari odds.
$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X$$
Nilai ini disebut **Logit**. Hubungan ini memastikan bahwa probabilitas $p$ akan selalu berada di antara 0 dan 1.

#### Studi Kasus: Deteksi Kanker

Dalam deteksi kanker, Recall (Sensitivity) jauh lebih penting daripada Precision. 
Lebih baik kita salah mendiagnosa orang sehat sebagai sakit (False Positive) agar dilakukan tes lanjutan, 
daripada melewatkan orang yang benar-benar sakit (False Negative) yang bisa berakibat fatal.

#### Koefisien Gini vs Entropi

Meskipun kita akan membahas Pohon Keputusan di bab berikutnya, konsep ketidakmurnian (*impurity*) mulai relevan di sini. 
Gini impurity mengukur seberapa sering elemen yang dipilih secara acak akan salah diklasifikasikan.

#### Teknik Evaluasi Tambahan: Lift Chart dan Gain Chart

Banyak digunakan dalam pemasaran untuk melihat seberapa jauh model klasifikasi lebih baik daripada pemilihan acak.

#### Ringkasan Metrik Evaluasi Klasifikasi

| Metrik | Fokus Utama | Kapan Digunakan |
| :--- | :--- | :--- |
| Akurasi | Kebenaran Total | Data Seimbang |
| Precision | Menghindari FP | Spam Filter, Prediksi Saham |
| Recall | Menghindari FN | Diagnosa Medis, Keamanan |
| AUC-ROC | Kemampuan Memisahkan Kelas | Perbandingan Model Umum |
| Kappa | Kesepakatan di luar kebetulan | Penilaian Subjektif |

#### Implementasi Praktis: Penanganan Outliers dalam Klasifikasi

Outliers dalam fitur dapat menggeser garis keputusan (decision boundary) pada model seperti Regresi Logistik dan LDA. 
Sangat penting untuk melakukan scaling dan penanganan outlier sebelum melatih model.

In [ ]:
from sklearn.preprocessing import RobustScaler

# RobustScaler menggunakan median dan IQR sehingga tidak peka terhadap outlier
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_train_b)
log_reg_robust = LogisticRegression().fit(X_scaled, y_train_b)
print(f"Koefisien Robust: {log_reg_robust.coef_}")

#### Penutup Akhir Bab 5

Klasifikasi adalah fondasi bagi banyak aplikasi AI modern. 
Di Bab 6, kita akan mengeksplorasi teknik Machine Learning Statistik yang lebih canggih, seperti Random Forest dan XGBoost.